# M1, Submodule 0 - Grasp a standalone brick, located via the wrist-mounted RGBD camera

The robot does not know where the brick is. The RealSense camera is mounted on the wrist (eye-in-hand, per the course's practical 4 setup), not fixed externally. A single brick is placed anywhere on the table; the arm moves to a taught "observe" joint configuration where the wrist camera looks down at the whole table, the RGBD point cloud is segmented (height above the table + horizontal bounds) to find the brick, and the grasp pose comes from that estimate. Mirrors `src/M1/simulation/submodule_0.py`.

Three one-time calibration/teaching steps make this possible and are cached to disk after the first run: the camera's pose relative to the TCP (eye-in-hand hand-eye calibration, via `airo_camera_toolkit.calibration`), the table height in the base frame (measured once by touching it with the gripper), and the observe joint configuration (taught once via freedrive, same pattern as the course's practical 4 viewpoint-teaching).

In [ ]:
import json
import socket
from pathlib import Path

import numpy as np
from airo_camera_toolkit.calibration.fiducial_markers import AIRO_DEFAULT_ARUCO_DICT, AIRO_DEFAULT_CHARUCO_BOARD
from airo_camera_toolkit.calibration.hand_eye_calibration import do_camera_robot_calibration
from airo_camera_toolkit.cameras.realsense.realsense import Realsense
from airo_camera_toolkit.point_clouds.operations import transform_point_cloud
from airo_dataset_tools.data_parsers.pose import Pose
from airo_robots.grippers.hardware.robotiq_2f85_urcap import Robotiq2F85
from airo_robots.manipulators.hardware.ur_rtde import URrtde

In [ ]:
ROBOT_IP = "10.42.0.162"
RTDE_PORT = 30004

TABLE_LENGTH = 0.80
TABLE_WIDTH = 0.60
ROBOT_OFFSET_X = 0.09
ROBOT_OFFSET_Y = 0.07

LIFT_HEIGHT = 0.15
PREGRASP_CLEARANCE = 0.12
PREGRASP_LINEAR_SPEED = 0.1
GRASP_LINEAR_SPEED = 0.03
LIFT_LINEAR_SPEED = 0.05

GRIPPER_CLOSE_SPEED = 0.02
GRIPPER_CLOSE_FORCE = 25

MIN_HEIGHT_ABOVE_TABLE = 0.002
MAX_HEIGHT_ABOVE_TABLE = 0.05
BASE_EXCLUSION_RADIUS = 0.15

DATA_DIR = Path.cwd().resolve().parents[1] / "data"
CALIBRATION_DIR = DATA_DIR / "M1_camera_calibration"
TABLE_HEIGHT_PATH = DATA_DIR / "M1_table_height.json"
OBSERVE_JOINTS_PATH = DATA_DIR / "M1_observe_joints.json"

In [ ]:
def robot_reachable(ip: str = ROBOT_IP, port: int = RTDE_PORT, timeout: float = 2.0) -> bool:
    try:
        with socket.create_connection((ip, port), timeout=timeout):
            return True
    except OSError:
        return False


if robot_reachable():
    print(f"{ROBOT_IP}:{RTDE_PORT} is reachable.")
else:
    print(f"Could not reach {ROBOT_IP}:{RTDE_PORT} within a few seconds.")
    print("Not on the robot's network yet")

In [ ]:
assert robot_reachable(), (
    f"Could not reach {ROBOT_IP}:{RTDE_PORT} - check you're on the robot's network and it's in "
    "remote control mode before running this cell."
)

robot = URrtde(ROBOT_IP, URrtde.UR3E_CONFIG)
print("Connected. Current joint configuration (rad):", np.round(robot.get_joint_configuration(), 3))

In [ ]:
gripper = Robotiq2F85(ROBOT_IP)
robot.gripper = gripper

print("Gripper active:", gripper.gripper_is_active())
print("Current finger width (m):", round(gripper.get_current_width(), 4))

In [ ]:
camera = Realsense()
print("Connected to RealSense camera. Resolution:", camera.resolution)
print("Intrinsics matrix:\n", camera.intrinsics_matrix())

## Hand-eye calibration (camera pose relative to the TCP)

One-time, cached to `CALIBRATION_DIR` after the first run. The camera is wrist-mounted, so this is **eye-in-hand** calibration: it finds the fixed transform from the TCP frame to the camera, not a fixed base-frame pose. Needs the AIRO default ChArUco board (7x5 squares, 4cm squares / 3.1cm markers, `DICT_4X4_250`) printed and held in view of the camera while the arm (in freedrive) moves through a few varied viewpoints.

In [ ]:
def find_best_calibration(calibration_dir: Path):
    results_dirs = sorted(calibration_dir.glob("results_n=*"), key=lambda p: int(p.name.split("=")[1]))
    if not results_dirs:
        return None
    results_dir = results_dirs[-1]
    errors_path = results_dir / "residual_errors.json"
    if not errors_path.exists():
        return None
    errors = json.loads(errors_path.read_text())
    best_method = min(errors, key=lambda k: errors[k])
    pose_path = results_dir / f"camera_pose_{best_method}.json"
    if not pose_path.exists():
        return None
    pose = Pose.model_validate_json(pose_path.read_text())
    return pose.as_homogeneous_matrix()


camera_pose_in_tcp = find_best_calibration(CALIBRATION_DIR)
if camera_pose_in_tcp is not None:
    print("Loaded existing camera calibration from", CALIBRATION_DIR)
else:
    print("No existing calibration found - run the next cell.")
print(camera_pose_in_tcp)

In [ ]:
if camera_pose_in_tcp is None:
    CALIBRATION_DIR.mkdir(parents=True, exist_ok=True)
    do_camera_robot_calibration(
        "eye_in_hand", AIRO_DEFAULT_ARUCO_DICT, AIRO_DEFAULT_CHARUCO_BOARD, camera, robot, str(CALIBRATION_DIR)
    )
    camera_pose_in_tcp = find_best_calibration(CALIBRATION_DIR)
    print(camera_pose_in_tcp)

## Table height in the robot's base frame

One-time, cached to `TABLE_HEIGHT_PATH`. This is a static environment measurement (like the simulation script knowing where it built the table), not the brick's location.

In [ ]:
if TABLE_HEIGHT_PATH.exists():
    table_height_in_base = json.loads(TABLE_HEIGHT_PATH.read_text())["z"]
    print("Loaded table height:", table_height_in_base)
else:
    print("Entering freedrive. Move the gripper down until it just touches the table surface.")
    robot.rtde_control.freedriveMode()
    input("Press Enter when touching the table...")
    robot.rtde_control.endFreedriveMode()

    table_height_in_base = float(robot.get_tcp_pose()[2, 3])
    TABLE_HEIGHT_PATH.parent.mkdir(parents=True, exist_ok=True)
    TABLE_HEIGHT_PATH.write_text(json.dumps({"z": table_height_in_base}))
    print("Measured and saved table height:", table_height_in_base)

## Observe pose (wrist camera looking at the whole table)

One-time, cached to `OBSERVE_JOINTS_PATH`. Same pattern as the course's practical 4 viewpoint-teaching: freedrive the arm so the wrist camera looks down at the table with the whole table in view, then record the joint configuration (not just the TCP pose - we replay this with `move_to_joint_configuration`, so the camera ends up in exactly the same pose every time).

In [ ]:
if OBSERVE_JOINTS_PATH.exists():
    observe_joints = np.array(json.loads(OBSERVE_JOINTS_PATH.read_text())["joints"])
    print("Loaded observe joint configuration:", observe_joints)
else:
    print("Entering freedrive. Move the arm so the wrist camera looks down at the whole table.")
    robot.rtde_control.freedriveMode()
    print("Use the next cell (re-run it as needed) to preview the camera view while freedriving.")

In [ ]:
if not OBSERVE_JOINTS_PATH.exists():
    import matplotlib.pyplot as plt

    plt.imshow(camera.get_rgb_image_as_int())
    plt.axis("off")
    plt.show()
    print("Re-run this cell to refresh the preview. Once the whole table is in view, run the next cell.")

In [ ]:
if not OBSERVE_JOINTS_PATH.exists():
    robot.rtde_control.endFreedriveMode()
    observe_joints = robot.get_joint_configuration()
    OBSERVE_JOINTS_PATH.parent.mkdir(parents=True, exist_ok=True)
    OBSERVE_JOINTS_PATH.write_text(json.dumps({"joints": observe_joints.tolist()}))
    print("Saved observe joint configuration:", observe_joints)

In [ ]:
robot.move_to_joint_configuration(observe_joints).wait()

point_cloud_in_camera = camera.get_colored_point_cloud()
X_B_TCP = robot.get_tcp_pose()
X_B_C = X_B_TCP @ camera_pose_in_tcp
point_cloud_in_base = transform_point_cloud(point_cloud_in_camera, X_B_C)

points = point_cloud_in_base.points
x, y, z = points[:, 0], points[:, 1], points[:, 2]
height_above_table = z - table_height_in_base

mask = (
    (height_above_table > MIN_HEIGHT_ABOVE_TABLE)
    & (height_above_table < MAX_HEIGHT_ABOVE_TABLE)
    & (x > -ROBOT_OFFSET_X)
    & (x < TABLE_LENGTH - ROBOT_OFFSET_X)
    & (y > -ROBOT_OFFSET_Y)
    & (y < TABLE_WIDTH - ROBOT_OFFSET_Y)
    & (np.hypot(x, y) > BASE_EXCLUSION_RADIUS)
)
candidate = points[mask]
print("candidate point count:", candidate.shape[0])
assert candidate.shape[0] >= 10, "Could not find a brick - check it's on the table and within the camera's view."

centroid = candidate.mean(axis=0)
xy = candidate[:, :2] - centroid[:2]
cov = xy.T @ xy
eigvals, eigvecs = np.linalg.eigh(cov)
principal = eigvecs[:, np.argmax(eigvals)]
brick_yaw = np.arctan2(principal[1], principal[0])
brick_x, brick_y = centroid[0], centroid[1]
print(f"Perceived brick at (x={brick_x:.3f}, y={brick_y:.3f}, yaw={brick_yaw:.2f})")

In [ ]:
def rotation_matrix_z(yaw: float) -> np.ndarray:
    c, s = np.cos(yaw), np.sin(yaw)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])


def make_pose(x: float, y: float, z: float, rotation: np.ndarray) -> np.ndarray:
    pose = np.eye(4)
    pose[:3, :3] = rotation
    pose[:3, 3] = [x, y, z]
    return pose


TOP_DOWN_ROTATION = np.array([[1, 0, 0], [0, -1, 0], [0, 0, -1]])
grasp_rotation = rotation_matrix_z(brick_yaw) @ TOP_DOWN_ROTATION

pregrasp_pose = make_pose(brick_x, brick_y, table_height_in_base + PREGRASP_CLEARANCE, grasp_rotation)
grasp_pose = make_pose(brick_x, brick_y, table_height_in_base, grasp_rotation)
lift_pose = make_pose(brick_x, brick_y, table_height_in_base + LIFT_HEIGHT, grasp_rotation)

In [ ]:
print("Opening gripper...")
robot.gripper.open().wait()

In [ ]:
print("Reaching towards the brick (pregrasp)...")
robot.move_linear_to_tcp_pose(pregrasp_pose, linear_speed=PREGRASP_LINEAR_SPEED).wait()

In [ ]:
print("Descending onto the brick...")
robot.move_linear_to_tcp_pose(grasp_pose, linear_speed=GRASP_LINEAR_SPEED).wait()

In [ ]:
print("Closing the gripper...")
robot.gripper.move(0.0, speed=GRIPPER_CLOSE_SPEED, force=GRIPPER_CLOSE_FORCE).wait()

grasped = robot.gripper.is_an_object_grasped()
print("Object grasped:", grasped)

In [ ]:
print(f"Lifting {LIFT_HEIGHT * 100:.0f}cm straight up...")
robot.move_linear_to_tcp_pose(lift_pose, linear_speed=LIFT_LINEAR_SPEED).wait()
print("Done.")

## 9. Optional: release, to reset for another attempt

In [ ]:
robot.move_linear_to_tcp_pose(grasp_pose, linear_speed=LIFT_LINEAR_SPEED).wait()
robot.gripper.open().wait()
print("Released.")